# Learning Objectives

* Reproduce the first image of the M87* black hole with a Pegasus workflow built from EHT public releases.
* Run three independent imaging pipelines (DIFMAP CLEAN, eht-imaging RML, SMILI RML) in four Apptainer containers.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the reconstructed images, closure statistics, and REx ring measurements.

# EHT M87* Imaging Workflow

A clean-room reproduction of the 2019 EHT M87* images using only the EHT Collaboration's public releases
(calibrated data `2019-D01-01`, imaging pipelines `2019-D01-02`). Per observation day, the three fiducial
pipelines image the data independently; their images are rendered, scored against the data, aggregated,
and measured with REx. See `README.md` and `SPEC.md` for the full description and the comparison with the
published results.

**Workflow Jobs:**
1. `fetch_data` - download the calibrated uvfits release and verify SHA-256 checksums
2. `run_difmap` - DIFMAP CLEAN, per day and band (lo, hi)
3. `run_ehtim` - eht-imaging RML, per day (lo+hi combined)
4. `run_smili` - SMILI RML, per day and band
5. `render_images` - per-day PNG panel of every reconstruction
6. `compute_stats` - closure-phase and log-closure-amplitude chi^2 per image
7. `aggregate_results` - CSV table and markdown report
8. `measure_ring_rex` - ring diameter, width, orientation and asymmetry of every image (REx)

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Input Data

There is nothing to prepare: the `fetch_data` job downloads the EHT calibrated data release at run time and
checks it against `data/checksums.txt`. The EHT pipeline scripts it runs are vendored in `eht-pipelines/`
(see `eht-pipelines/PROVENANCE.md`).

## 2. Build the Apptainer Containers

The pipelines need four images, one per tool stack (`Apptainer/eht-{difmap,ehtim,smili,rex}.def`). Building
them can take a while; existing `.sif` files are kept.

In [ ]:
%%bash
for name in ehtim difmap smili rex; do
    if [ ! -f Apptainer/eht-${name}.sif ]; then
        apptainer build Apptainer/eht-${name}.sif Apptainer/eht-${name}.def
    else
        echo "Container already built: Apptainer/eht-${name}.sif"
    fi
done

## 3. Generate the Workflow

The following cell uses the `EHTWorkflow` class from `workflow_generator.py` — the same generator used by the
command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs) and
writes it to `workflow.yml`. Nothing about the workflow is defined here: change the pipeline in
`workflow_generator.py`, and both the notebook and the CLI pick it up.

This run images one day (April 11, day `101`) with all three pipelines — 10 jobs, about 10 minutes on a small
HTCondor pool. Set `DAYS = None` for all four days (31 jobs).

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import EHTWorkflow

DAGFILE = "workflow.yml"
EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

DAYS = ["101"]              # None = all four days (095, 096, 100, 101)
SMILI_NPROC = 2
SIF_DIR = str(Path("Apptainer").resolve())

workflow = EHTWorkflow(dagfile=DAGFILE, days=DAYS, smili_nproc=SMILI_NPROC, sif_dir=SIF_DIR)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nEHT M87 workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. `fetch_data` feeds every imaging job;
the per-day images fan in to the render, statistics and REx jobs.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog
chosen above.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/`:

| Output | Description |
|--------|-------------|
| `difmap_{day}_{band}.fits`, `ehtim_{day}.fits`, `smili_{day}_{band}.fits` | Reconstructed images |
| `m87_{day}_panel.png` | Per-day panel of every reconstruction |
| `stats_{day}.json`, `difmap_{day}_{band}.stat` | Closure chi^2 statistics |
| `stats_summary.csv`, `summary_report.md` | Aggregated statistics and report |
| `ring_rex.csv` | REx ring measurements of every image |
| `checksums_manifest.txt` | SHA-256 of the fetched data, as verified |

In [ ]:
!ls -ltR output/

The reconstructed images for the day we ran — the ring of the M87* black hole shadow, as each pipeline sees it:

In [ ]:
DAY = (DAYS or ["101"])[0]

Image(filename=f"output/m87_{DAY}_panel.png")

And the ring measurements — compare the diameters with the EHT's published ~42 µas:

In [ ]:
!cat output/ring_rex.csv